# M01-P01 — Exploring Bedrock LLMs

Trabajaremos directamente con **Amazon Bedrock Converse API**, sin LangChain. El objetivo es abrir la caja y observar qué enviamos al modelo, qué devuelve y cómo cambian los resultados al modificar `system prompt`, temperatura, `maxTokens`, contexto y streaming.

> Si la sesión SSO ha caducado: `aws sso login --profile genai-course --use-device-code`


## 1. Cliente e identidad


In [ ]:
import json
import time

import boto3
import pandas as pd

from enterprise_genai.config import get_settings

settings = get_settings()

session = boto3.Session(
    profile_name=settings.aws_profile or None,
    region_name=settings.aws_region,
)

identity = session.client("sts").get_caller_identity()
runtime = session.client(
    "bedrock-runtime",
    region_name=settings.aws_region,
)

MODEL_ID = settings.bedrock_text_model_id

print("AWS identity:", identity["Arn"])
print("Region:", settings.aws_region)
print("Model:", MODEL_ID)


## 2. Primera llamada — abrir la caja

Construimos explícitamente la petición para ver sus piezas principales: `modelId`, `system`, `messages` e `inferenceConfig`.


In [ ]:
incident = (
    "Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. "
    "El servicio web corporativo funciona correctamente. "
    "El equipo de red informa de errores intermitentes en el backend de autenticación."
)

request = {
    "modelId": MODEL_ID,
    "system": [
        {
            "text": (
                "Eres un asistente de soporte empresarial. "
                "Resume incidencias de forma clara y no inventes datos."
            )
        }
    ],
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": f"Resume esta incidencia en 3 viñetas:\n\n{incident}"
                }
            ],
        }
    ],
    "inferenceConfig": {
        "temperature": 0.2,
        "maxTokens": 200,
    },
}

print(json.dumps(request, indent=2, ensure_ascii=False))

response = runtime.converse(**request)

print("\n" + response["output"]["message"]["content"][0]["text"])
print("\nStop reason:", response.get("stopReason"))
print("Usage:", response.get("usage"))
print("Metrics:", response.get("metrics"))


### Qué debes observar

- `inputTokens`: tokens enviados al modelo.
- `outputTokens`: tokens generados.
- `totalTokens`: consumo total.
- `latencyMs`: tiempo de inferencia reportado por Bedrock.
- `stopReason`: por qué terminó la generación.

Estas métricas nos permitirán hablar más adelante de coste, rendimiento y observabilidad usando datos reales.


## 3. Banco de pruebas

La siguiente función evita repetir código mientras mantiene visibles los parámetros principales de Converse.


In [ ]:
def run_experiment(
    label,
    user_message,
    *,
    system_prompt=None,
    temperature=0.2,
    max_tokens=200,
    model_id=MODEL_ID,
):
    kwargs = {
        "modelId": model_id,
        "messages": [
            {
                "role": "user",
                "content": [{"text": user_message}],
            }
        ],
        "inferenceConfig": {
            "temperature": temperature,
            "maxTokens": max_tokens,
        },
    }

    if system_prompt:
        kwargs["system"] = [{"text": system_prompt}]

    started = time.perf_counter()
    response = runtime.converse(**kwargs)
    elapsed_ms = round((time.perf_counter() - started) * 1000)

    usage = response.get("usage", {})
    metrics = response.get("metrics", {})

    return {
        "label": label,
        "text": response["output"]["message"]["content"][0]["text"],
        "input_tokens": usage.get("inputTokens"),
        "output_tokens": usage.get("outputTokens"),
        "bedrock_latency_ms": metrics.get("latencyMs"),
        "client_elapsed_ms": elapsed_ms,
        "stop_reason": response.get("stopReason"),
    }


## 4. Experimento A — `system prompt`

Aislamos una sola variable: el `system prompt`. El mensaje del usuario fija el formato para evitar que la longitud contamine el experimento.


In [ ]:
user_message = (
    "Analiza esta incidencia. Responde exactamente en 3 líneas, "
    "sin introducción ni conclusión:\n"
    "1. Impacto: máximo 20 palabras.\n"
    "2. Explicación más probable: máximo 20 palabras.\n"
    "3. Siguiente acción: máximo 20 palabras.\n\n"
    + incident
)

variants = [
    ("Sin system prompt", None),
    (
        "Coordinador NOC",
        (
            "Actúa como coordinador de un NOC empresarial. "
            "No afirmes una causa raíz si no está demostrada. "
            "Distingue claramente entre hechos observados e hipótesis."
        ),
    ),
]

results = []

for label, system_prompt in variants:
    r = run_experiment(
        label,
        user_message,
        system_prompt=system_prompt,
        temperature=0.2,
        max_tokens=220,
    )
    results.append(r)

    print(f"\n--- {label} ---")
    print(r["text"])
    print("Stop reason:", r["stop_reason"])

pd.DataFrame(results)[
    [
        "label",
        "input_tokens",
        "output_tokens",
        "bedrock_latency_ms",
        "stop_reason",
    ]
]


### Observa

Los datos y el formato son los mismos. Compara si cambia la prudencia de la respuesta: ¿distingue hechos de hipótesis?, ¿afirma una causa no demostrada?, ¿cambia la acción recomendada?

**Modifica** el segundo `system prompt` y conviértelo en un ejecutivo, un técnico N1 o un responsable de seguridad.


## 5. Experimento B — temperatura

Comparamos `temperature=0.0` y `temperature=0.8` repitiendo tres veces la misma tarea. Valores bajos suelen favorecer estabilidad; valores altos aumentan la probabilidad de variación, pero no obligan a que cada salida sea diferente.


In [ ]:
temp_results = []

temperature_prompt = (
    "Genera exactamente 3 títulos alternativos para esta incidencia.\n"
    "Reglas:\n"
    "- un título por línea;\n"
    "- máximo 8 palabras por título;\n"
    "- sin explicaciones, recomendaciones ni introducción;\n"
    "- no inventes información.\n\n"
    + incident
)

for temperature in [0.0, 0.8]:
    for run in range(1, 4):
        r = run_experiment(
            f"T={temperature} run={run}",
            temperature_prompt,
            system_prompt=(
                "Eres un redactor técnico de un centro de operaciones. "
                "Cumple exactamente el formato solicitado."
            ),
            temperature=temperature,
            max_tokens=120,
        )

        titles = [
            line.strip()
            for line in r["text"].splitlines()
            if line.strip()
        ]

        temp_results.append(
            {
                "temperature": temperature,
                "run": run,
                "titulo_1": titles[0] if len(titles) > 0 else "",
                "titulo_2": titles[1] if len(titles) > 1 else "",
                "titulo_3": titles[2] if len(titles) > 2 else "",
                "tokens": r["output_tokens"],
                "latencia_ms": r["bedrock_latency_ms"],
                "stop_reason": r["stop_reason"],
            }
        )

temperature_df = pd.DataFrame(temp_results)

pd.set_option("display.max_colwidth", None)
temperature_df


### Observa

Con temperatura 0 las ejecuciones deberían tender a parecerse mucho. Con 0.8 puede aparecer más diversidad en palabras y estructuras. La temperatura es probabilística: no es un interruptor de “creatividad”.


## 6. Experimento C — `maxTokens`

Ahora sí queremos provocar deliberadamente truncamiento. El prompt fija una respuesta acotada y cambiamos únicamente el límite máximo de salida.


In [ ]:
length_results = []

length_prompt = (
    "Redacta exactamente 5 líneas, una para cada apartado:\n"
    "Resumen, Impacto, Evidencias, Hipótesis y Próximos pasos.\n"
    "Reglas:\n"
    "- máximo 18 palabras por línea;\n"
    "- no añadas introducción ni conclusión;\n"
    "- no inventes información;\n"
    "- si un dato no consta, indícalo.\n\n"
    + incident
)

for max_tokens in [40, 300]:
    r = run_experiment(
        f"maxTokens={max_tokens}",
        length_prompt,
        system_prompt=(
            "Eres un analista de incidencias. "
            "Distingue hechos de hipótesis y no inventes datos."
        ),
        temperature=0.2,
        max_tokens=max_tokens,
    )

    length_results.append(r)

    print(f"\n--- maxTokens={max_tokens} ---")
    print(r["text"])
    print("Stop reason:", r["stop_reason"])

pd.DataFrame(length_results)[
    [
        "label",
        "output_tokens",
        "bedrock_latency_ms",
        "stop_reason",
    ]
]


### Observa

El límite pequeño debería acabar en `max_tokens` y dejar una respuesta incompleta. El límite amplio debería permitir que el modelo termine con `end_turn`.

`maxTokens` es un **límite duro**, no una instrucción de estilo ni una longitud objetivo.


## 7. Experimento D — contexto

Todavía no hacemos RAG. Añadimos manualmente un fragmento de información empresarial y mantenemos iguales el modelo, la pregunta, el `system prompt`, la temperatura y `maxTokens`. La única variable es la presencia del contexto.


In [ ]:
question = (
    "Para un P1 de VPN de un cliente premium, "
    "¿cuál es el tiempo objetivo de respuesta y qué equipo coordina?"
)

system_prompt = (
    "Responde de forma breve, en máximo 3 líneas. "
    "No inventes políticas corporativas. "
    "Si no dispones de información suficiente, indícalo claramente."
)

context = (
    "Procedimiento OPS-INC-001:\n"
    "- Cliente premium + P1: respuesta inicial en 30 minutos.\n"
    "- La coordinación inicial corresponde al NOC.\n"
    "- Seguridad participa si hay indicadores de compromiso."
)

without_context = run_experiment(
    "Sin contexto",
    question,
    system_prompt=system_prompt,
    temperature=0.0,
    max_tokens=100,
)

with_context = run_experiment(
    "Con contexto",
    f"Contexto corporativo:\n\n{context}\n\nPregunta: {question}",
    system_prompt=system_prompt,
    temperature=0.0,
    max_tokens=100,
)

for r in [without_context, with_context]:
    print(f"\n--- {r['label']} ---")
    print(r["text"])
    print("Stop reason:", r["stop_reason"])

pd.DataFrame([without_context, with_context])[
    [
        "label",
        "input_tokens",
        "output_tokens",
        "bedrock_latency_ms",
        "stop_reason",
    ]
]


La idea que reutilizaremos en RAG es:

`pregunta + contexto relevante → respuesta fundamentada`

Más adelante RAG automatizará la búsqueda del contexto que aquí hemos añadido manualmente.


## 8. Experimento E — streaming

Ahora estudiamos **cómo llega** la respuesta. Medimos cuándo está disponible el stream, cuándo llega el primer fragmento de texto y cuándo termina toda la respuesta.


In [ ]:
stream_request = {
    "modelId": MODEL_ID,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Responde SOLO con 3 viñetas.\n"
                        "- máximo 10 palabras por viñeta;\n"
                        "- sin título, introducción ni conclusión;\n"
                        "- explica por qué un fallo de autenticación VPN "
                        "puede afectar a muchos usuarios."
                    )
                }
            ],
        }
    ],
    "inferenceConfig": {
        "temperature": 0.2,
        "maxTokens": 120,
    },
}

started = time.perf_counter()

response = runtime.converse_stream(**stream_request)

stream_ready_ms = round(
    (time.perf_counter() - started) * 1000
)

first_chunk_ms = None
stop_reason = None
usage = None
metrics = None

for event in response["stream"]:
    if "contentBlockDelta" in event:
        delta = event["contentBlockDelta"]["delta"]

        if "text" in delta:
            if first_chunk_ms is None:
                first_chunk_ms = round(
                    (time.perf_counter() - started) * 1000
                )

            print(delta["text"], end="", flush=True)

    if "messageStop" in event:
        stop_reason = event["messageStop"].get("stopReason")

    if "metadata" in event:
        usage = event["metadata"].get("usage")
        metrics = event["metadata"].get("metrics")

total_ms = round(
    (time.perf_counter() - started) * 1000
)

response_metadata = response.get("ResponseMetadata", {})

print("\n")
print("Stream ready (ms):", stream_ready_ms)
print("First text chunk (ms):", first_chunk_ms)
print("Total client time (ms):", total_ms)
print("Stop reason:", stop_reason)
print("Usage:", usage)
print("Bedrock metrics:", metrics)
print("Retry attempts:", response_metadata.get("RetryAttempts"))


### Observa

El streaming no tiene por qué reducir la latencia total, pero permite que el usuario empiece a ver contenido antes. Compara `First text chunk` con `Total client time`.

Si alguna ejecución presenta una espera anómala, compara también `Bedrock metrics` y `Retry attempts`: la latencia percibida por el cliente puede incluir factores externos a la inferencia.


## 9. Mini-reto — actualización ejecutiva

Cerramos la práctica combinando restricciones de formato, temperatura baja y control de alucinaciones. La respuesta debe utilizar hechos conocidos y separar claramente una recomendación de los datos observados.


In [ ]:
executive = run_experiment(
    "Director de operaciones",
    (
        "Prepara una actualización ejecutiva a partir de esta incidencia.\n\n"
        "Devuelve exactamente 3 líneas:\n"
        "1. Situación: máximo 22 palabras.\n"
        "2. Impacto conocido: máximo 22 palabras.\n"
        "3. Siguiente acción recomendada: máximo 22 palabras.\n\n"
        + incident
    ),
    system_prompt=(
        "Actúa como asistente de un director de operaciones. "
        "En Situación e Impacto usa únicamente hechos presentes en la incidencia. "
        "No estimes porcentajes, número total de usuarios, pérdidas económicas, "
        "severidad, SLA ni alcance no indicado. "
        "No afirmes una causa raíz. "
        "En la tercera línea puedes proponer una acción, pero debe quedar claro "
        "que es una recomendación."
    ),
    temperature=0.1,
    max_tokens=180,
)

print(executive["text"])
print("\nTokens:", executive["input_tokens"], "+", executive["output_tokens"])
print("Latency:", executive["bedrock_latency_ms"], "ms")
print("Stop reason:", executive["stop_reason"])


## Cierre

Hemos observado directamente:

`system prompt + user message + contexto + parámetros → Bedrock Converse → respuesta + tokens + latencia + stopReason`

Ideas clave:

- una respuesta plausible puede contener afirmaciones no sustentadas;
- un `system prompt` puede cambiar el comportamiento sin cambiar los datos;
- la temperatura modifica la probabilidad de variación, no garantiza creatividad ni determinismo;
- `maxTokens` es un techo de generación;
- el contexto empresarial permite respuestas fundamentadas;
- el streaming mejora la latencia percibida.

En M01-P02 pasaremos de texto libre a **salidas estructuradas** y construiremos la primera funcionalidad visible de `Analyze`.
